# 2. OpenAI の チャット API の基礎


## 2.3. 入出力の長さの制限や料金に影響する「トークン」


### トークン


In [1]:
!pip install tiktoken==0.7.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 18.4 MB/s eta 0:00:00
  Attempting uninstall: tiktoken
    Found existing installation: tiktoken 0.12.0
    Uninstalling tiktoken-0.12.0:
      Successfully uninstalled tiktoken-0.12.0


In [2]:
import tiktoken

text = "ChatGPT"

encoding = tiktoken.encoding_for_model("gpt-4o")
tokens = encoding.encode(text)
for token in tokens:
    print(encoding.decode([token]))

Chat
GPT


### Tokenizer と tiktoken の紹介


In [4]:
import tiktoken

text = "LLMを使ってクールなものを作るのは簡単だが、プロダクションで使えるものを作るのは非常に難しい。"

encoding = tiktoken.encoding_for_model("gpt-4o")
tokens = encoding.encode(text)
print(len(tokens))

37


## 2.4. Chat Completions API を試す環境の準備


### OpenAI の API キーの準備


In [5]:
import os
from google.colab import userdata

os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

## 2.5. Chat Completions API のハンズオン


### OpenAI のライブラリ


#### 【注意】既知のエラーについて

openai パッケージが依存する httpx のアップデートにより、`openai==1.40.6` を使用する箇所で `TypeError: Client.__init__() got an unexpected keyword argument 'proxies'` というエラーが発生するようになりました。

このエラーは、`!pip install httpx==0.27.2` のように、httpx の特定バージョンをインストールすることで回避することができます。

なお、Google Colab で一度上記のエラーに遭遇したあとで `!pip install httpx==0.27.2` のようにパッケージをインストールし直した場合、以下のどちらかの操作を実施する必要があります。

- Google Colab の「ランタイム」から「セッションを再起動する」を実行する
- 「ランタイムを接続解除して削除」を実行してパッケージのインストールからやり直す


In [6]:
!pip install openai==1.40.6 httpx==0.27.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 361.3/361.3 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 7.4 MB/s eta 0:00:00
  Attempting uninstall: httpx
    Found existing installation: httpx 0.28.1
    Uninstalling httpx-0.28.1:
      Successfully uninstalled httpx-0.28.1
  Attempting uninstall: openai
    Found existing installation: openai 2.28.0
    Uninstalling openai-2.28.0:
      Successfully uninstalled openai-2.28.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
firebase-admin 6.9.0 requires httpx[http2]==0.28.1, but you have httpx 0.27.2 which is incompatible.
google-genai 1.67.0 requires httpx<1.0.0,>=0.28.1, but you have httpx 0.27.2 which is incompatible.


### Chat Completions API の呼び出し


In [9]:
from openai import OpenAI

client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

response = client.chat.completions.create(
    model="gemini-3.1-flash-lite-preview",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "こんにちは！私はジョンと言います！"},
    ],
)
print(response.to_json(indent=2))

{
  "id": "ch7GafWFIaCX-8YP9sfW4A0",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "message": {
        "content": "こんにちは、ジョンさん！はじめまして。お会いできて嬉しいです。\n私はAIアシスタントです。何かお手伝いできることはありますか？気軽になんでも話しかけてくださいね！",
        "role": "assistant",
        "extra_content": {
          "google": {
            "thought_signature": "EjQKMgG+Pvb72YsPm3lAPOJSWGPZkq+YL5geanMtuWLmpYGfhgh3pBzgovdVs7qN5v7b7luM"
          }
        }
      }
    }
  ],
  "created": 1774591602,
  "model": "gemini-3.1-flash-lite-preview",
  "object": "chat.completion",
  "usage": {
    "completion_tokens": 40,
    "prompt_tokens": 15,
    "total_tokens": 55
  }
}


### 会話履歴を踏まえた応答を得る


In [10]:
response = client.chat.completions.create(
    model="gemini-3.1-flash-lite-preview",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "こんにちは！私はジョンと言います！"},
        {"role": "assistant", "content": "こんにちは、ジョンさん！お会いできて嬉しいです。今日はどんなことをお話ししましょうか？"},
        {"role": "user", "content": "私の名前が分かりますか？"},
    ],
)
print(response.to_json(indent=2))

{
  "id": "tx_GaZOTM42rjrEP-PK6oQM",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "message": {
        "content": "はい、もちろんです！先ほど「ジョン」さんと教えてくださいましたね。覚えていますよ。",
        "role": "assistant",
        "extra_content": {
          "google": {
            "thought_signature": "EjQKMgG+Pvb7pFhtFVUARH1bZeyZue6Y6Jn9pqx2jSKC9TbgPW5kzYy7AX++WCWpLpEzrQqe"
          }
        }
      }
    }
  ],
  "created": 1774591927,
  "model": "gemini-3.1-flash-lite-preview",
  "object": "chat.completion",
  "usage": {
    "completion_tokens": 21,
    "prompt_tokens": 43,
    "total_tokens": 64
  }
}


### ストリーミングで応答を得る


In [11]:
response = client.chat.completions.create(
    model="gemini-3.1-flash-lite-preview",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "こんにちは！私はジョンと言います！"},
    ],
    stream=True,
)

for chunk in response:
    content = chunk.choices[0].delta.content
    if content is not None:
        print(content, end="", flush=True)

ジョンさん、こんにちは！はじめまして。お会いできて嬉しいです。
私はAIアシスタントです。何かお手伝いできることはありますか？気軽になんでも話しかけてくださいね！

### JSON モード


In [12]:
from openai import OpenAI

client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

response = client.chat.completions.create(
    model="gemini-3.1-flash-lite-preview",
    messages=[
        {
            "role": "system",
            "content": '人物一覧を次のJSON形式で出力してください。\n{"people": ["aaa", "bbb"]}',
        },
        {
            "role": "user",
            "content": "昔々あるところにおじいさんとおばあさんがいました",
        },
    ],
    response_format={"type": "json_object"},
)
print(response.choices[0].message.content)

{"people": ["おじいさん", "おばあさん"]}


### Vision（画像入力）


In [14]:
from openai import OpenAI

client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

image_url = "https://raw.githubusercontent.com/yoshidashingo/langchain-book/main/assets/cover.jpg"

response = client.chat.completions.create(
    model="gemini-3.1-flash-lite-preview",
    messages=[
        {
            "role": "user",
            "content": [
                {"type": "text", "text": "画像を説明してください。"},
                {"type": "image_url", "image_url": {"url": image_url}},
            ],
        }
    ],
)

print(response.choices[0].message.content)

この画像は、『ChatGPT/LangChainによるチャットシステム構築［実践］入門』という技術専門書の表紙です。

主な特徴は以下の通りです。

*   **タイトル:** 『ChatGPT/LangChainによるチャットシステム構築［実践］入門』
*   **著者:** 吉田真吾、大嶋勇樹
*   **出版社:** 技術評論社（エンジニア選書）
*   **デザイン:** 鮮やかな青緑色の背景に、鎖と止まり木にとまったオウムのイラストが描かれています。これは「LangChain（Lang=言語、Chain=鎖）」というライブラリ名にちなんだものと思われます。
*   **内容:** 大規模言語モデル（LLM）を活用したシステム開発に関する解説書です。以下の内容が含まれていることが記載されています。
    *   OpenAI API（Chat Completions API）の活用
    *   LangChainの基礎から実践的な解説
    *   Slackチャットアプリの開発チュートリアル
    *   本番リリースに向けた備え
    *   すべてクラウドサービスで完結する構成

ChatGPTとLangChainを使って、実際に動作するチャットシステムを構築したい開発者向けの学習用書籍です。


### （コラム）Completions API


In [ ]:
from openai import OpenAI

client = OpenAI()

response = client.completions.create(
    model="gpt-3.5-turbo-instruct",
    prompt="こんにちは！私はジョンと言います！",
)
print(response.to_json(indent=2))

## 2.6. Function calling


### Function calling のサンプルコード


In [15]:
import json


def get_current_weather(location, unit="fahrenheit"):
    if "tokyo" in location.lower():
        return json.dumps({"location": "Tokyo", "temperature": "10", "unit": unit})
    elif "san francisco" in location.lower():
        return json.dumps(
            {"location": "San Francisco", "temperature": "72", "unit": unit}
        )
    elif "paris" in location.lower():
        return json.dumps({"location": "Paris", "temperature": "22", "unit": unit})
    else:
        return json.dumps({"location": location, "temperature": "unknown"})

In [16]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_weather",
            "description": "Get the current weather in a given location",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city and state, e.g. San Francisco, CA",
                    },
                    "unit": {"type": "string", "enum": ["celsius", "fahrenheit"]},
                },
                "required": ["location"],
            },
        },
    }
]

In [23]:
from openai import OpenAI

client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

messages = [
    {"role": "user", "content": "Tokyoの天気はどうですか？"},
]

response = client.chat.completions.create(
    model="gemini-3.1-flash-lite-preview",
    messages=messages,
    tools=tools,
    tool_choice="auto"
)
print(response.to_json(indent=2))

{
  "id": "ESrGaeiiEseEjrEPjNi0sAY",
  "choices": [
    {
      "finish_reason": "tool_calls",
      "index": 0,
      "message": {
        "role": "assistant",
        "tool_calls": [
          {
            "id": "2h9suxpj",
            "function": {
              "arguments": "{\"location\":\"Tokyo, Japan\"}",
              "name": "get_current_weather"
            },
            "type": "function",
            "extra_content": {
              "google": {
                "thought_signature": "EjQKMgG+Pvb7FmRGcyKvV3ud3i2rP1j+O+9lIJqQL4I+R6Q0ulNFSVAQYotNKeWMNHsURywe"
              }
            }
          }
        ]
      }
    }
  ],
  "created": 1774594577,
  "model": "gemini-3.1-flash-lite-preview",
  "object": "chat.completion",
  "usage": {
    "completion_tokens": 20,
    "prompt_tokens": 98,
    "total_tokens": 118
  }
}


In [24]:
response_message = response.choices[0].message
messages.append(response_message.to_dict())

In [25]:
available_functions = {
    "get_current_weather": get_current_weather,
}

# 使いたい関数は複数あるかもしれないのでループ
for tool_call in response_message.tool_calls:
    # 関数を実行
    function_name = tool_call.function.name
    function_to_call = available_functions[function_name]
    function_args = json.loads(tool_call.function.arguments)
    function_response = function_to_call(
        location=function_args.get("location"),
        unit=function_args.get("unit"),
    )
    print(function_response)

    # 関数の実行結果を会話履歴としてmessagesに追加
    messages.append(
        {
            "tool_call_id": tool_call.id,
            "role": "tool",
            "name": function_name,
            "content": function_response,
        }
    )

{"location": "Tokyo", "temperature": "10", "unit": null}


In [26]:
print(json.dumps(messages, ensure_ascii=False, indent=2))

[
  {
    "role": "user",
    "content": "Tokyoの天気はどうですか？"
  },
  {
    "role": "assistant",
    "tool_calls": [
      {
        "id": "2h9suxpj",
        "function": {
          "arguments": "{\"location\":\"Tokyo, Japan\"}",
          "name": "get_current_weather"
        },
        "type": "function",
        "extra_content": {
          "google": {
            "thought_signature": "EjQKMgG+Pvb7FmRGcyKvV3ud3i2rP1j+O+9lIJqQL4I+R6Q0ulNFSVAQYotNKeWMNHsURywe"
          }
        }
      }
    ]
  },
  {
    "tool_call_id": "2h9suxpj",
    "role": "tool",
    "name": "get_current_weather",
    "content": "{\"location\": \"Tokyo\", \"temperature\": \"10\", \"unit\": null}"
  }
]


In [27]:
second_response = client.chat.completions.create(
    model="gemini-3.1-flash-lite-preview",
    messages=messages,
)
print(second_response.to_json(indent=2))

{
  "id": "TSrGafS-OfvPjrEPobrI6Q8",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "message": {
        "content": "現在、東京の気温は10℃です。\n\nもし詳しい予報（天気や降水確率など）を知りたい場合は、お住まいの市区町村などを教えていただければ、より正確な情報をお調べします。",
        "role": "assistant",
        "extra_content": {
          "google": {
            "thought_signature": "EjQKMgG+Pvb7aScjjBPgyR0ggY/mevbFENK1+E3wbHXT0k37B9D6Ec4xiQVTdjVLSzttKGiR"
          }
        }
      }
    }
  ],
  "created": 1774594637,
  "model": "gemini-3.1-flash-lite-preview",
  "object": "chat.completion",
  "usage": {
    "completion_tokens": 47,
    "prompt_tokens": 60,
    "total_tokens": 107
  }
}
